In [3]:
# ============================================
# Colab-ready: Final version (curves only, y-axis 0-1.2, no vertical lines)
# Generates:
#   - ./results_real/fig_dephasing.jpg
#   - ./results_real/fig_depolarizing.jpg
#   - ./results_real/fig_amplitude_damping.jpg
#   - ./results_real/panel_channels_real.jpg (3 subplots in one figure)
#
#
# For single-qubit states we use the standard hemispherical representation
# associated with Bures geometry, in which geodesics are represented by
# great-circle arcs in an embedded R^4 picture. GAC is evaluated from the
# angle between the tangent of the step geodesic A->B and that of the
# reference geodesic A->T at the same point A.
#For a state with Bloch vector r, the embedding is:
#
#     q(r) = (sqrt(1 - ||r||^2), rx, ry, rz)  in R^4
#
# Under this map, Bures geodesics become great-circle arcs on the
# hemisphere and their tangent vectors are obtained in closed form via
# standard spherical geometry.  GAC is then the cosine of the angle
# between the tangent of the step geodesic A->B and the tangent of the
# reference geodesic A->T at the same point A, evaluated as a plain
# Euclidean dot product in R^4 (the global scale factor of the metric
# cancels in the cosine).
#
# This approach is exact for d=2.  For d >= 3 geodesics on the Bures
# manifold are not reducible to great circles and would require either
# numerical integration of the geodesic ODE or optimisation over
# purifications -- noted as a current limitation of the framework.
# ============================================

import os, shutil
import numpy as np
import pandas as pd
from numpy.random import default_rng
from scipy.linalg import sqrtm
import matplotlib.pyplot as plt

# -----------------------------
# Clean output folder
# -----------------------------
if os.path.exists('./results_real'):
    shutil.rmtree('./results_real')
os.makedirs('./results_real', exist_ok=True)

# -----------------------------
# Configuration
# -----------------------------
N_POINTS     = 61
DEPTH        = 32
N_RANDOM     = 24
rng = default_rng(42)

# Experimental ranges (Sec. 5.1.2)
RANGE_DEPH = (0.005, 0.03)
RANGE_DEPO = (0.015, 0.075)
RANGE_AD   = (0.80, 1.00)

# -----------------------------
# Linear algebra + channels
# -----------------------------
I2 = np.eye(2, dtype=np.complex128)
X  = np.array([[0, 1],[1, 0]], dtype=np.complex128)
Y  = np.array([[0, -1j],[1j, 0]], dtype=np.complex128)
Z  = np.array([[1, 0],[0, -1]], dtype=np.complex128)

def pure_to_rho(psi):
    psi = np.asarray(psi, dtype=np.complex128).reshape(2,1)
    psi = psi / np.linalg.norm(psi)
    return psi @ psi.conj().T

def haar_random_pure_qubit(n: int):
    states = []
    for _ in range(n):
        z = rng.normal(size=2) + 1j * rng.normal(size=2)
        psi = z / np.linalg.norm(z)
        states.append(pure_to_rho(psi))
    return states

def fidelity(rho, sigma):
    sr = sqrtm(rho)
    inner = sr @ sigma @ sr
    inner = (inner + inner.conj().T) / 2.0
    root = sqrtm(inner)
    val = np.real(np.trace(root))**2
    return float(np.clip(val, 0.0, 1.0))

def bures_distance(rho, sigma):
    F = fidelity(rho, sigma)
    return float(np.sqrt(max(0.0, 2.0*(1.0 - np.sqrt(F)))))

def apply_kraus(rho, Ks):
    out = np.zeros((2,2), dtype=np.complex128)
    for K in Ks:
        out += K @ rho @ K.conj().T
    out = (out + out.conj().T) / 2.0
    tr = np.real(np.trace(out))
    return out / tr if tr != 0 else I2/2

def dephasing_kraus(p):        return [np.sqrt(1-p)*I2, np.sqrt(p)*Z]
def depolarizing_kraus(p):
    a0 = np.sqrt(1 - 3*p/4); a = np.sqrt(p/4)
    return [a0*I2, a*X, a*Y, a*Z]
def amplitude_damping_kraus(g):
    K0 = np.array([[1,0],[0,np.sqrt(1-g)]], dtype=np.complex128)
    K1 = np.array([[0,np.sqrt(g)],[0,0]], dtype=np.complex128)
    return [K0, K1]

def trajectory(rho0, kraus_fn, param, depth):
    traj = [rho0]
    Ks = kraus_fn(param)
    rho = rho0
    for _ in range(depth):
        rho = apply_kraus(rho, Ks)
        traj.append(rho)
    return traj

def trajectory_length_bures(traj):
    return sum(bures_distance(traj[i], traj[i+1]) for i in range(len(traj)-1))

def gdi(traj):
    LB = trajectory_length_bures(traj)
    DB = bures_distance(traj[0], traj[-1])
    return float(LB / max(DB, 1e-12))

# -----------------------------
# GAC via hemispheric embedding
# -----------------------------
# The Bures manifold of mixed qubit states is isometric to a hemisphere
# of S^3 (unit sphere in R^4).  The embedding map sends a state rho with
# Bloch vector r = (rx, ry, rz), ||r|| <= 1, to the unit vector:
#
#     q(r) = ( sqrt(1 - ||r||^2),  rx,  ry,  rz )  in R^4
#
# Key properties exploited here:
#   * Bures geodesics <-> great-circle arcs on this hemisphere.
#   * The initial tangent of the great circle from q0 to q1 is:
#       v = (q1 - (q0 . q1) * q0) / ||q1 - (q0 . q1) * q0||
#     (standard spherical logarithmic map on S^n).
#   * The angle between two tangents at q0 equals their Euclidean angle
#     in R^4, so GAC = dot(v_AB, v_AT) with both vectors unit-normalised.
#     The global scale factor of the Bures metric (1/4) cancels in the
#     cosine and does not affect the result.

def rho_to_bloch(rho):
    """Extract the Bloch vector (rx, ry, rz) from a 2x2 density matrix."""
    rx = np.real(np.trace(rho @ X))
    ry = np.real(np.trace(rho @ Y))
    rz = np.real(np.trace(rho @ Z))
    return np.array([rx, ry, rz], dtype=float)

def bloch_to_bures_point(r):
    """Embed a Bloch vector into the S^3 hemisphere: q = (sqrt(1-||r||^2), r)."""
    nr2 = float(np.dot(r, r))
    nr2 = min(max(nr2, 0.0), 1.0)       # clip for numerical safety
    return np.array([np.sqrt(1.0 - nr2), r[0], r[1], r[2]], dtype=float)

def great_circle_tangent(q0, q1, eps=1e-12):
    """Initial unit tangent of the great-circle geodesic from q0 to q1 on S^3.

    Uses the spherical logarithmic map:
        v = (q1 - c * q0) / ||q1 - c * q0||,   c = q0 . q1

    Returns the zero vector in degenerate cases (e.g. coincident states or
    numerically ill-conditioned configurations).
    """
    c = float(np.clip(np.dot(q0, q1), -1.0, 1.0))
    v = q1 - c * q0
    nv = np.linalg.norm(v)
    if nv < eps:
        return np.zeros(4, dtype=float)
    return v / nv

# Notation correspondence with the paper:
#   A = current state rho_i
#   B = next state rho_{i+1}
#   T = final state of the trajectory
#   qA, qB, qT = embedded representations in R^4
#   vAB = tangent direction associated with the local step geodesic A -> B
#   vAT = tangent direction associated with the reference geodesic A -> T
#
# In the notation of the manuscript, vAB plays the role of the local tangent
# direction of the channel-induced trajectory \dot{\gamma}(t), while vAT
# corresponds to the tangent direction of the reference geodesic \dot{\tau}(t).
# Since GAC is a normalised cosine, only directions are needed.
def gac_step(A, B, T, eps=1e-12):
    """GAC contribution at step A->B relative to the target T.

    Returns the cosine of the angle between the initial tangent of the
    Bures geodesic A->B and that of the Bures geodesic A->T, both
    computed as great-circle tangents on the S^3 hemisphere.
    """
    qA = bloch_to_bures_point(rho_to_bloch(A))
    qB = bloch_to_bures_point(rho_to_bloch(B))
    qT = bloch_to_bures_point(rho_to_bloch(T))

    vAB = great_circle_tangent(qA, qB, eps=eps)
    vAT = great_circle_tangent(qA, qT, eps=eps)

    # Degenerate case: if the step is null or the current state already equals
    # the target, the tangent direction is undefined; we assign GAC = 1.0 by convention.

    if np.linalg.norm(vAB) < eps or np.linalg.norm(vAT) < eps:
        return 1.0   # degenerate step: treat as perfectly aligned

    return float(np.clip(np.dot(vAB, vAT), -1.0, 1.0))

def gac_mean(traj):
    """Geodesic Alignment Coefficient (REVISED - Comment 4).

    Computes the mean cosine of the angle between:
      - the initial tangent of the Bures geodesic traj[i] -> traj[i+1], and
      - the initial tangent of the Bures geodesic traj[i] -> T  (endpoint),
    using the exact great-circle representation on the S^3 hemisphere.

    This replaces the previous law-of-cosines heuristic, which approximated
    the angle from scalar Bures distances rather than from tangent vectors,
    and was not a proper Riemannian alignment measure.
    """
    if len(traj) < 2:
        return 1.0
    T = traj[-1]
    vals = [gac_step(traj[i], traj[i+1], T) for i in range(len(traj) - 1)]
    return float(np.mean(vals)) if vals else 1.0

# -----------------------------
# States
# -----------------------------
def sample_states(n_random: int):
    CARDINAL = [
        pure_to_rho([1,0]), pure_to_rho([0,1]),
        pure_to_rho([1/np.sqrt(2),  1/np.sqrt(2)]),
        pure_to_rho([1/np.sqrt(2), -1/np.sqrt(2)]),
        pure_to_rho([1/np.sqrt(2),  1j/np.sqrt(2)]),
        pure_to_rho([1/np.sqrt(2), -1j/np.sqrt(2)]),
    ]
    return CARDINAL + haar_random_pure_qubit(n_random)

# -----------------------------
# Sweep runner
# -----------------------------
def run_sweep_means(channel_name, kraus_fn, grid, depth, n_random):
    rows = []
    states = sample_states(n_random)
    for param in grid:
        Fs,DBs,GDIs,GACs = [],[],[],[]
        for rho0 in states:
            traj = trajectory(rho0, kraus_fn, float(param), depth)
            Fs.append(fidelity(traj[0], traj[-1]))
            DBs.append(bures_distance(traj[0], traj[-1]))
            GDIs.append(gdi(traj))
            GACs.append(gac_mean(traj))
        rows.append({
            'channel': channel_name,
            'param': float(param),
            'F':   float(np.mean(Fs)),
            'DB':  float(np.mean(DBs)),
            'GDI': float(np.mean(GDIs)),
            'GAC': float(np.mean(GACs)),
        })
    return pd.DataFrame(rows)

# -----------------------------
# Build grids & run sweeps
# -----------------------------
grid_deph = np.linspace(*RANGE_DEPH, N_POINTS)
grid_depo = np.linspace(*RANGE_DEPO, N_POINTS)
grid_ad   = np.linspace(*RANGE_AD,   N_POINTS)

df_deph = run_sweep_means('dephasing',        dephasing_kraus,         grid_deph, DEPTH, N_RANDOM)
df_depo = run_sweep_means('depolarizing',     depolarizing_kraus,      grid_depo, DEPTH, N_RANDOM)
df_ad   = run_sweep_means('amplitude_damping',amplitude_damping_kraus, grid_ad,   DEPTH, N_RANDOM)

# -----------------------------
# Plot individual figures (Y axis fixed 0-1.2)
# -----------------------------
def plot_channel_curves(df, title, xlabel, outpath):
    d = df.sort_values('param')
    plt.figure()
    plt.plot(d['param'], d['F'],   label='Fidelity')
    plt.plot(d['param'], d['DB'],  label='Bures distance')
    plt.plot(d['param'], d['GDI'], label='GDI')
    plt.plot(d['param'], d['GAC'], label='GAC')
    plt.xlabel(xlabel)
    plt.ylabel('Index value')
    plt.title(title)
    plt.ylim(0, 1.2)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, format='jpg')
    plt.close()

plot_channel_curves(df_deph, 'Dephasing channel',        'Noise parameter p',  './results_real/fig_dephasing.jpg')
plot_channel_curves(df_depo, 'Depolarizing channel',     'Noise parameter p',  './results_real/fig_depolarizing.jpg')
plot_channel_curves(df_ad,   'Amplitude damping channel','Damping parameter g', './results_real/fig_amplitude_damping.jpg')

# -----------------------------
# Panel with 3 subplots (shared y-axis 0-1.2)
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)

# Left: Dephasing
d = df_deph.sort_values('param')
axes[0].plot(d['param'], d['F'],   label='Fidelity')
axes[0].plot(d['param'], d['DB'],  label='Bures distance')
axes[0].plot(d['param'], d['GDI'], label='GDI')
axes[0].plot(d['param'], d['GAC'], label='GAC')
axes[0].set_title('Dephasing')
axes[0].set_xlabel('Noise parameter p')
axes[0].set_ylabel('Index value')
axes[0].set_ylim(0, 1.2)
axes[0].grid(alpha=0.3)

# Middle: Depolarizing
d = df_depo.sort_values('param')
axes[1].plot(d['param'], d['F'])
axes[1].plot(d['param'], d['DB'])
axes[1].plot(d['param'], d['GDI'])
axes[1].plot(d['param'], d['GAC'])
axes[1].set_title('Depolarizing')
axes[1].set_xlabel('Noise parameter p')
axes[1].set_ylim(0, 1.2)
axes[1].grid(alpha=0.3)

# Right: Amplitude damping (no vertical lines)
d = df_ad.sort_values('param')
axes[2].plot(d['param'], d['F'])
axes[2].plot(d['param'], d['DB'])
axes[2].plot(d['param'], d['GDI'])
axes[2].plot(d['param'], d['GAC'])
axes[2].set_title('Amplitude damping')
axes[2].set_xlabel('Damping parameter g')
axes[2].set_ylim(0, 1.2)
axes[2].grid(alpha=0.3)

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.subplots_adjust(bottom=0.18)
plt.savefig('./results_real/panel_channels_real.jpg', dpi=300, format='jpg')
plt.close()

print('Saved final figures in ./results_real as JPG 300dpi with y-axis [0,1.2] and no vertical lines.')
print('GAC computed via exact great-circle geodesics on the S^3 hemispheric embedding of qubit state space.')


/tmp/ipykernel_2482/3577814125.py:82: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  sr = sqrtm(rho)
/tmp/ipykernel_2482/3577814125.py:85: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  root = sqrtm(inner)


Saved final figures in ./results_real as JPG 300dpi with y-axis [0,1.2] and no vertical lines.
GAC computed via exact great-circle geodesics on the S^3 hemispheric embedding of qubit state space.


In [4]:
# ============================================
# Colab-ready: Final version (curves only, y-axis 0-1.2, no vertical lines)
# Generates:
#   - ./results_real/fig_dephasing.jpg
#   - ./results_real/fig_depolarizing.jpg
#   - ./results_real/fig_amplitude_damping.jpg
#   - ./results_real/panel_channels_real.jpg (3 subplots in one figure)
#
#
# For single-qubit states we use the standard hemispherical representation
# associated with Bures geometry, in which geodesics are represented by
# great-circle arcs in an embedded R^4 picture. GAC is evaluated from the
# angle between the tangent of the step geodesic A->B and that of the
# reference geodesic A->T at the same point A.
#For a state with Bloch vector r, the embedding is:
#
#     q(r) = (sqrt(1 - ||r||^2), rx, ry, rz)  in R^4
#
# Under this map, Bures geodesics become great-circle arcs on the
# hemisphere and their tangent vectors are obtained in closed form via
# standard spherical geometry.  GAC is then the cosine of the angle
# between the tangent of the step geodesic A->B and the tangent of the
# reference geodesic A->T at the same point A, evaluated as a plain
# Euclidean dot product in R^4 (the global scale factor of the metric
# cancels in the cosine).
#
# This approach is exact for d=2.  For d >= 3 geodesics on the Bures
# manifold are not reducible to great circles and would require either
# numerical integration of the geodesic ODE or optimisation over
# purifications -- noted as a current limitation of the framework.
# ============================================

import os, shutil
import numpy as np
import pandas as pd
from numpy.random import default_rng
from scipy.linalg import sqrtm
import matplotlib.pyplot as plt

# -----------------------------
# Clean output folder
# -----------------------------
if os.path.exists('./results_baseline'):
    shutil.rmtree('./results_baseline')
os.makedirs('./results_baseline', exist_ok=True)

# -----------------------------
# Configuration
# -----------------------------
N_POINTS = 101          # resolution of [0,1] sweep
DEPTH    = 33           # odd depth to avoid parity artifacts
N_RANDOM = 24           # number of Haar-random initial states
EPS      = 1e-6         # avoid exact 0 and 1
rng = default_rng(42)

# -----------------------------
# Linear algebra + channels
# -----------------------------
I2 = np.eye(2, dtype=np.complex128)
X  = np.array([[0, 1],[1, 0]], dtype=np.complex128)
Y  = np.array([[0, -1j],[1j, 0]], dtype=np.complex128)
Z  = np.array([[1, 0],[0, -1]], dtype=np.complex128)

def pure_to_rho(psi):
    psi = np.asarray(psi, dtype=np.complex128).reshape(2,1)
    psi = psi / np.linalg.norm(psi)
    return psi @ psi.conj().T

def haar_random_pure_qubit(n: int):
    states = []
    for _ in range(n):
        z = rng.normal(size=2) + 1j*rng.normal(size=2)
        psi = z / np.linalg.norm(z)
        states.append(pure_to_rho(psi))
    return states

def fidelity(rho, sigma):
    sr = sqrtm(rho)
    inner = sr @ sigma @ sr
    inner = (inner + inner.conj().T) / 2.0
    root  = sqrtm(inner)
    val   = np.real(np.trace(root))**2
    return float(np.clip(val, 0.0, 1.0))

def bures_distance(rho, sigma):
    F = fidelity(rho, sigma)
    return float(np.sqrt(max(0.0, 2.0*(1.0 - np.sqrt(F)))))

def apply_kraus(rho, Ks):
    out = np.zeros((2,2), dtype=np.complex128)
    for K in Ks:
        out += K @ rho @ K.conj().T
    out = (out + out.conj().T) / 2.0
    tr  = np.real(np.trace(out))
    return out / tr if tr != 0 else I2/2

# --- Channels ---

# Dephasing (phase-damping): attenuates coherences without unitary flip
def dephasing_kraus(p):
    K0 = np.sqrt(1 - p) * I2
    K1 = np.sqrt(p) * np.array([[1,0],[0,0]], dtype=np.complex128)  # |0><0|
    K2 = np.sqrt(p) * np.array([[0,0],[0,1]], dtype=np.complex128)  # |1><1|
    return [K0, K1, K2]

# Depolarizing (Pauli form, valid for p in [0,1])
def depolarizing_kraus(p):
    return [np.sqrt(1.0 - p)*I2,
            np.sqrt(p/3.0)*X,
            np.sqrt(p/3.0)*Y,
            np.sqrt(p/3.0)*Z]

# Amplitude damping: gamma in [0,1]
def amplitude_damping_kraus(g):
    K0 = np.array([[1,0],[0,np.sqrt(1-g)]], dtype=np.complex128)
    K1 = np.array([[0,np.sqrt(g)],[0,0]], dtype=np.complex128)
    return [K0, K1]

# -----------------------------
# Trajectories & classical indices
# -----------------------------
def trajectory(rho0, kraus_fn, param, depth):
    traj = [rho0]
    Ks = kraus_fn(param)
    rho = rho0
    for _ in range(depth):
        rho = apply_kraus(rho, Ks)
        traj.append(rho)
    return traj

def trajectory_length_bures(traj):
    return sum(bures_distance(traj[i], traj[i+1]) for i in range(len(traj)-1))

def gdi(traj, eps=1e-9):
    LB = trajectory_length_bures(traj)
    DB = bures_distance(traj[0], traj[-1])
    return float(LB / max(DB, eps))

# -----------------------------
# GAC via hemispheric embedding
# -----------------------------
# The Bures manifold of mixed qubit states is isometric to a hemisphere
# of S^3 (unit sphere in R^4).  The embedding map sends a state rho with
# Bloch vector r = (rx, ry, rz), ||r|| <= 1, to the unit vector:
#
#     q(r) = ( sqrt(1 - ||r||^2),  rx,  ry,  rz )  in R^4
#
# Key properties exploited here:
#   * Bures geodesics <-> great-circle arcs on this hemisphere.
#   * The initial tangent of the great circle from q0 to q1 is:
#       v = (q1 - (q0 . q1) * q0) / ||q1 - (q0 . q1) * q0||
#     (standard spherical logarithmic map on S^n).
#   * The angle between two tangents at q0 equals their Euclidean angle
#     in R^4, so GAC = dot(v_AB, v_AT) with both vectors unit-normalised.
#     The global scale factor of the Bures metric (1/4) cancels in the
#     cosine and does not affect the result.

def rho_to_bloch(rho):
    """Extract the Bloch vector (rx, ry, rz) from a 2x2 density matrix."""
    rx = np.real(np.trace(rho @ X))
    ry = np.real(np.trace(rho @ Y))
    rz = np.real(np.trace(rho @ Z))
    return np.array([rx, ry, rz], dtype=float)

def bloch_to_bures_point(r):
    """Embed a Bloch vector into the S^3 hemisphere: q = (sqrt(1-||r||^2), r)."""
    nr2 = float(np.dot(r, r))
    nr2 = min(max(nr2, 0.0), 1.0)       # clip for numerical safety
    return np.array([np.sqrt(1.0 - nr2), r[0], r[1], r[2]], dtype=float)

def great_circle_tangent(q0, q1, eps=1e-12):
    """Initial unit tangent of the great-circle geodesic from q0 to q1 on S^3.

    Uses the spherical logarithmic map:
        v = (q1 - c * q0) / ||q1 - c * q0||,   c = q0 . q1

    Returns the zero vector in degenerate cases (e.g. coincident states or
    numerically ill-conditioned configurations).
    """
    c = float(np.clip(np.dot(q0, q1), -1.0, 1.0))
    v = q1 - c * q0
    nv = np.linalg.norm(v)
    if nv < eps:
        return np.zeros(4, dtype=float)
    return v / nv

# Notation correspondence with the paper:
#   A = current state rho_i
#   B = next state rho_{i+1}
#   T = final state of the trajectory
#   qA, qB, qT = embedded representations in R^4
#   vAB = tangent direction associated with the local step geodesic A -> B
#   vAT = tangent direction associated with the reference geodesic A -> T
#
# In the notation of the manuscript, vAB plays the role of the local tangent
# direction of the channel-induced trajectory \dot{\gamma}(t), while vAT
# corresponds to the tangent direction of the reference geodesic \dot{\tau}(t).
# Since GAC is a normalised cosine, only directions are needed.
def gac_step(A, B, T, eps=1e-12):
    """GAC contribution at step A->B relative to the target T.

    Returns the cosine of the angle between the initial tangent of the
    Bures geodesic A->B and that of the Bures geodesic A->T, both
    computed as great-circle tangents on the S^3 hemisphere.
    """
    qA = bloch_to_bures_point(rho_to_bloch(A))
    qB = bloch_to_bures_point(rho_to_bloch(B))
    qT = bloch_to_bures_point(rho_to_bloch(T))

    vAB = great_circle_tangent(qA, qB, eps=eps)
    vAT = great_circle_tangent(qA, qT, eps=eps)

    # Degenerate case: if the step is null or the current state already equals
    # the target, the tangent direction is undefined; we assign GAC = 1.0 by convention.
    if np.linalg.norm(vAB) < eps or np.linalg.norm(vAT) < eps:
        return 1.0   # degenerate step: treat as perfectly aligned

    return float(np.clip(np.dot(vAB, vAT), -1.0, 1.0))

def gac_mean(traj):
    """Geodesic Alignment Coefficient (REVISED - Comment 4).

    Computes the mean cosine of the angle between:
      - the initial tangent of the Bures geodesic traj[i] -> traj[i+1], and
      - the initial tangent of the Bures geodesic traj[i] -> T  (endpoint),
    using the exact great-circle representation on the S^3 hemisphere.

    This replaces the previous law-of-cosines heuristic, which approximated
    the angle from scalar Bures distances rather than from tangent vectors,
    and was not a proper Riemannian alignment measure.
    """
    if len(traj) < 2:
        return 1.0
    T = traj[-1]
    vals = [gac_step(traj[i], traj[i+1], T) for i in range(len(traj) - 1)]
    return float(np.mean(vals)) if vals else 1.0

# -----------------------------
# States (Haar only)
# -----------------------------
def sample_states(n_random: int):
    return haar_random_pure_qubit(n_random)

# -----------------------------
# Sweep runner (means)
# -----------------------------
def run_sweep_means(channel_name, kraus_fn, grid, depth, n_random):
    rows = []
    states = sample_states(n_random)
    for param in grid:
        Fs, DBs, GDIs, GACs = [], [], [], []
        for rho0 in states:
            traj = trajectory(rho0, kraus_fn, float(param), depth)
            Fs.append(fidelity(traj[0], traj[-1]))
            DBs.append(bures_distance(traj[0], traj[-1]))
            GDIs.append(gdi(traj))
            GACs.append(gac_mean(traj))
        rows.append({
            'channel': channel_name,
            'param': float(param),
            'F':   float(np.mean(Fs)),
            'DB':  float(np.mean(DBs)),
            'GDI': float(np.mean(GDIs)),
            'GAC': float(np.mean(GACs)),
        })
    return pd.DataFrame(rows)

# -----------------------------
# Build grids [0,1] (avoid exact endpoints) & run sweeps
# -----------------------------
grid_01 = np.linspace(0.0 + EPS, 1.0 - EPS, N_POINTS)

df_deph = run_sweep_means('dephasing',        dephasing_kraus,         grid_01, DEPTH, N_RANDOM)
df_depo = run_sweep_means('depolarizing',     depolarizing_kraus,      grid_01, DEPTH, N_RANDOM)
df_ad   = run_sweep_means('amplitude_damping',amplitude_damping_kraus, grid_01, DEPTH, N_RANDOM)

# -----------------------------
# Plot helpers (y in [0,2])
# -----------------------------
def plot_channel_curves(df, title, xlabel, outpath):
    d = df.sort_values('param')
    plt.figure()
    plt.plot(d['param'], d['F'],   label='Fidelity')
    plt.plot(d['param'], d['DB'],  label='Bures distance')
    plt.plot(d['param'], d['GDI'], label='GDI')
    plt.plot(d['param'], d['GAC'], label='GAC')
    plt.xlabel(xlabel)
    plt.ylabel('Index value')
    plt.title(title)
    plt.ylim(0, 2)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, format='jpg')
    plt.close()

plot_channel_curves(df_deph, 'Dephasing channel',        'Noise parameter p',  './results_baseline/fig_dephasing.jpg')
plot_channel_curves(df_depo, 'Depolarizing channel',     'Noise parameter p',  './results_baseline/fig_depolarizing.jpg')
plot_channel_curves(df_ad,   'Amplitude damping channel','Damping parameter g', './results_baseline/fig_amplitude_damping.jpg')

# -----------------------------
# Panel (3 subplots, shared y=[0,2])
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)

# Dephasing
d = df_deph.sort_values('param')
axes[0].plot(d['param'], d['F'],   label='Fidelity')
axes[0].plot(d['param'], d['DB'],  label='Bures distance')
axes[0].plot(d['param'], d['GDI'], label='GDI')
axes[0].plot(d['param'], d['GAC'], label='GAC')
axes[0].set_title('Dephasing')
axes[0].set_xlabel('Noise parameter p')
axes[0].set_ylabel('Index value')
axes[0].set_ylim(0, 2)
axes[0].grid(alpha=0.3)

# Depolarizing
d = df_depo.sort_values('param')
axes[1].plot(d['param'], d['F'])
axes[1].plot(d['param'], d['DB'])
axes[1].plot(d['param'], d['GDI'])
axes[1].plot(d['param'], d['GAC'])
axes[1].set_title('Depolarizing')
axes[1].set_xlabel('Noise parameter p')
axes[1].set_ylim(0, 2)
axes[1].grid(alpha=0.3)

# Amplitude damping
d = df_ad.sort_values('param')
axes[2].plot(d['param'], d['F'])
axes[2].plot(d['param'], d['DB'])
axes[2].plot(d['param'], d['GDI'])
axes[2].plot(d['param'], d['GAC'])
axes[2].set_title('Amplitude damping')
axes[2].set_xlabel('Damping parameter g')
axes[2].set_ylim(0, 2)
axes[2].grid(alpha=0.3)

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.subplots_adjust(bottom=0.18)
plt.savefig('./results_baseline/panel_channels_baseline.jpg', dpi=300, format='jpg')
plt.close()

print('Saved 0-1 sweep figures in ./results_baseline (JPG 300dpi, y=[0,2], phase-damping, DEPTH=33, eps guards).')
print('GAC computed via exact great-circle geodesics on the S^3 hemispheric embedding of qubit state space.')


/tmp/ipykernel_2482/239785167.py:81: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  root  = sqrtm(inner)
/tmp/ipykernel_2482/239785167.py:78: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  sr = sqrtm(rho)


Saved 0-1 sweep figures in ./results_baseline (JPG 300dpi, y=[0,2], phase-damping, DEPTH=33, eps guards).
GAC computed via exact great-circle geodesics on the S^3 hemispheric embedding of qubit state space.
